# Exploração quantitativa

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_04/01_exploracao_quantitativa.ipynb)

O guia definiu que explorar exige separar cálculo, interpretação e hipótese. Este
notebook inicia essa prática pela classificação das variáveis, porque a escala de
cada uma determina quais resumos e comparações são defensáveis.

## Tipos de variáveis

![Escalas nominal, ordinal, quantitativa e temporal são relacionadas a operações e gráficos; identificadores aparecem separados das medidas.](imagens/01_tipos_variaveis.svg)

Nominais distinguem; ordinais ordenam; quantitativas discretas contam; contínuas medem. Datas e identificadores têm papéis próprios. `dtype` não determina a escala conceitual.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_04'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from IPython.display import display
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from graficos import distribuicao_anotada

dados = pd.read_csv("dados/documentos.csv")
pd.DataFrame(
    [["genero", "nominal"], ["ano", "temporal"],
     ["palavras", "quantitativa discreta"], ["id_documento", "identificador"]],
    columns=["variavel", "escala"],
)

A classificação anterior determina quais operações são conceitualmente
adequadas. Para variáveis nominais, como `tema` e `genero`, começaremos descrevendo
quantos documentos pertencem a cada categoria e que parcela do corpus representam.

## Frequências e proporções

Se $x_i$ é a categoria do documento $i$, a frequência absoluta da categoria
$k$ e sua proporção são:

$$
f_k = \sum_{i=1}^{n}\mathbf{1}(x_i=k),
\qquad
p_k = \frac{f_k}{n}.
$$

| Símbolo | Significado | Operação em Python |
|---|---|---|
| $n$ | total de documentos incluídos | `len(dados)` |
| $f_k$ | documentos cuja categoria é $k$ | `value_counts()` |
| $p_k$ | parcela do total na categoria $k$ | `value_counts(normalize=True)` |

A função indicadora $\mathbf{1}(x_i=k)$ vale 1 quando o documento pertence à
categoria e 0 caso contrário. Declare sempre $n$: documentos não medem
automaticamente intensidade, importância histórica ou quantidade de menções.

In [ ]:
frequencias_tema = dados["tema"].value_counts().rename("frequencia")
proporcoes_tema = dados["tema"].value_counts(normalize=True).rename("proporcao")
pd.concat([frequencias_tema, proporcoes_tema], axis=1)

Frequências e proporções resumem categorias. Quando a variável registra uma
quantidade, como o número de palavras, precisamos primeiro perguntar onde os valores
se concentram. Essa pergunta introduz as medidas de tendência central.

## Medidas de tendência central

Medidas de tendência central resumem uma distribuição por um valor considerado
central ou típico. Elas não são intercambiáveis.

### Média

Para valores quantitativos $x_1,\ldots,x_n$, a média aritmética é a soma dividida
pelo número de observações:

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i.
$$

A média funciona como ponto de equilíbrio da distribuição e usa todos os valores.
Por isso, é sensível a valores extremos. Ela é adequada quando somar os valores e
dividir igualmente o total entre as unidades possui interpretação substantiva. Não
faz sentido calcular a média de identificadores ou de categorias nominais.

### Mediana

A mediana é o valor central depois de ordenar as observações. Se $n$ for par, usa-se
a média dos dois valores centrais:

$$
\widetilde{x}=
\begin{cases}
x_{((n+1)/2)}, & n \text{ ímpar},\\[4pt]
\dfrac{x_{(n/2)}+x_{(n/2+1)}}{2}, & n \text{ par}.
\end{cases}
$$

Ela depende da posição, não da distância de todos os valores ao centro, e tende a
ser menos sensível a extremos. É especialmente informativa em distribuições
assimétricas, mas não substitui a inspeção da distribuição.

### Moda

A moda é o valor ou categoria de maior frequência. Pode ser usada também com
variáveis nominais, para as quais média e mediana não são adequadas. Uma distribuição
pode ter uma moda, várias modas ou nenhuma moda informativa. Se todos os valores
aparecem uma vez, o `pandas` devolve todos em `mode()`; escolher apenas o primeiro
criaria uma falsa moda única.

| Medida | Pergunta resumida | Sensibilidade ou limite |
|---|---|---|
| média | qual seria a parcela igual do total por unidade? | usa todos os valores e responde fortemente a extremos |
| mediana | qual valor ocupa o centro da ordenação? | não expressa as distâncias entre todos os casos |
| moda | qual valor ou categoria ocorre mais vezes? | pode ser múltipla ou não informativa |

In [ ]:
x = dados["palavras"]
frequencias_palavras = x.value_counts()
frequencia_maxima = int(frequencias_palavras.max())
valores_modais = sorted(frequencias_palavras[frequencias_palavras.eq(frequencia_maxima)].index.tolist())
moda_informativa = valores_modais if frequencia_maxima > 1 else "nenhuma: todos os valores ocorrem uma vez"

resumo_tendencia = pd.Series({
    "n": x.count(),
    "media": x.mean(),
    "mediana": x.median(),
    "frequencia_da_moda": frequencia_maxima,
    "moda_informativa": moda_informativa,
})

display(distribuicao_anotada(
    x.tolist(), dados["id_documento"].tolist(), x.mean(), x.median()
))
resumo_tendencia

A média e a mediana localizam o centro; a moda examina repetição. Nenhuma delas
informa quanto os valores se afastam entre si. Para distinguir distribuições com o
mesmo centro, precisamos acrescentar medidas de dispersão.

## Medidas de dispersão

A **amplitude** é a distância entre máximo e mínimo. Usa apenas os dois extremos:

$$
A=x_{\max}-x_{\min}.
$$

Os quartis $Q_1$ e $Q_3$ delimitam a metade central dos valores ordenados. O intervalo
interquartil é:

$$
IQR=Q_3-Q_1.
$$

A variância amostral calcula a média corrigida dos desvios quadráticos em relação à
média; o desvio-padrão retorna à unidade original:

$$
s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2,
\qquad
s=\sqrt{s^2}.
$$

Variância fica em unidades ao quadrado e é menos intuitiva para descrição direta. O
desvio-padrão fica na mesma unidade de $x$, mas ambos são sensíveis a extremos. IQR é
mais resistente porque depende da metade central. O denominador $n-1$ corresponde a
`ddof=1`; para descrever uma população integral com denominador $n$, declare `ddof=0`.

In [ ]:
q1 = x.quantile(0.25, interpolation="linear")
q3 = x.quantile(0.75, interpolation="linear")
resumo_dispersao = pd.Series({
    "minimo": x.min(),
    "maximo": x.max(),
    "amplitude": x.max() - x.min(),
    "q1": q1,
    "q3": q3,
    "iqr": q3 - q1,
    "variancia_amostral": x.var(ddof=1),
    "desvio_padrao_amostral": x.std(ddof=1),
})
resumo_dispersao

As medidas de dispersão quantificam afastamentos, mas ainda não mostram onde
cada observação aparece nem por que um caso se afasta. A próxima etapa usa quartis e
IQR para localizar candidatos à inspeção na distribuição.

## Distribuição e valores extremos

Retome o $IQR$ calculado na seção anterior: ele cobre a metade central dos
valores ordenados e fornece uma regra convencional para localizar casos afastados.

A regra usada pelo boxplot define dois limites:

$$
L_{\mathrm{inferior}}=Q_1-1{,}5\,IQR,
\qquad
L_{\mathrm{superior}}=Q_3+1{,}5\,IQR.
$$

Um caso fora desses limites é um candidato à inspeção, nunca uma exclusão
automática ou prova de erro. Quartis possuem convenções de cálculo diferentes;
neste notebook registramos explicitamente a interpolação linear usada pelo pandas.

In [ ]:
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

extremos = dados[
    (dados["palavras"] < limite_inferior)
    | (dados["palavras"] > limite_superior)
]
print("Limites:", limite_inferior, "a", limite_superior)
extremos[["id_documento", "palavras", "genero", "tema"]]

Até aqui descrevemos uma variável por vez. Para investigar se a composição
dos temas muda entre gêneros documentais, precisamos cruzar duas variáveis
categóricas e tornar explícito o denominador de cada comparação.

## Tabela de contingência

Se $A$ representa o gênero e $B$ o tema, a célula $n_{ij}$ conta os
documentos que pertencem simultaneamente à linha $i$ e à coluna $j$:

$$
n_{ij}=\sum_{r=1}^{n}\mathbf{1}(A_r=i \land B_r=j).
$$

A proporção por linha usa como denominador o total daquela linha:

$$
p_{j\mid i}=\frac{n_{ij}}{\sum_j n_{ij}}.
$$

Assim, contagens e proporções por linha respondem perguntas diferentes.
`normalize="index"` implementa $p_{j\mid i}$. Um padrão descritivo não é teste,
explicação causal ou evidência automática de associação histórica.

In [ ]:
contagens = pd.crosstab(dados["genero"], dados["tema"])
proporcoes_por_genero = pd.crosstab(
    dados["genero"], dados["tema"], normalize="index"
).round(3)
contagens, proporcoes_por_genero

A tabela de contingência encerra o movimento que começou na classificação das
variáveis: escala, pergunta e denominador precisam permanecer coerentes. A
atividade reúne essas decisões antes que os resultados sejam levados para a
exploração textual e para as visualizações.

## Atividade

Classifique variáveis; escolha e justifique ao menos uma medida de tendência central
e uma de dispersão; declare denominadores; inspecione extremo e contingência.
Separe descrição, interpretação e hipótese. Ao concluir, registre
quais resultados merecem ser comparados com o conteúdo dos textos no Notebook 02.
Escreva aqui.